## a) Basic setup
1. python packages
2. API keys

In [ ]:
!pip install -q biopython requests pandas dotenv

In [ ]:
import os

# Detect environment and load secrets
try:
    from google.colab import userdata

    # --- Google Colab ---
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    ENTREZ_EMAIL = userdata.get("ENTREZ_EMAIL")
    NCBI_API_KEY = userdata.get("NCBI_API_KEY")

    print("Running in Google Colab.")

except ImportError:
    # --- Local Jupyter ---
    from dotenv import load_dotenv

    load_dotenv()  # reads .env from the notebook's directory

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
    ENTREZ_EMAIL = os.getenv("ENTREZ_EMAIL")
    NCBI_API_KEY = os.getenv("NCBI_API_KEY")

    print("Running locally.")

# Validate required secrets
assert GEMINI_API_KEY, (
    "GEMINI_API_KEY is not configured -- get a free key at https://aistudio.google.com/api-keys"
)

assert ENTREZ_EMAIL, (
    "ENTREZ_EMAIL is not configured."
)

if NCBI_API_KEY:
    print("NCBI_API_KEY found.")
else:
    print("No NCBI_API_KEY found -- using the slower NCBI rate limit.")


## b) Where the corpus comes from: query design before triage

Before any LLM triage runs, someone had to decide which papers to pull from PubMed/PMC in the first place. That decision — the search query and year range — is a data-quality lever in its own right, upstream of and arguably as consequential as which LLM you triage with.



In [ ]:
AMR_TERMS = (
    '("antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract] '
    'OR "drug resistance, microbial"[MeSH Terms] OR "multidrug resistance"[Title/Abstract] '
    'OR "multidrug resistant"[Title/Abstract] OR "resistance gene"[Title/Abstract] '
    'OR "resistant gene"[Title/Abstract] OR "resistant genes"[Title/Abstract] '
    'OR "resistance genes"[Title/Abstract] OR "antimicrobial susceptibility"[Title/Abstract])'
)
#OR "beta-lactamase"[Title/Abstract] OR "efflux pump"[Title/Abstract] 
#OR "minimum inhibitory concentration"[Title/Abstract] '


PUBTYPE_EXCLUSIONS = (
    'NOT "review"[pt] NOT "case reports"[pt] NOT "letter"[pt] NOT "comment"[pt] '
    'NOT "address"[pt] NOT "autobiography"[pt] NOT "biography"[pt] '
    'NOT "conference proceedings"[pt] NOT "editorial"[pt] NOT "interactive tutorial"[pt] '
    'NOT "interview"[pt] NOT "introductory journal article"[pt] '
    'NOT "personal narrative"[pt] NOT "portrait"[pt] NOT "news"[pt]'
)

# Optional terms to include
VARIANT_EXPERIMENT_TERMS = (
    '(("mutation"[MeSH Terms] OR "mutation"[All Fields] OR "mutations"[All Fields]) '
    'OR mutational[All Fields] OR mutant[All Fields] OR mutants[All Fields] '
    'OR variant[All Fields] OR variants[All Fields]) '
    'AND (experiment[All Fields] OR experimental[All Fields] OR "gene knockout"[All Fields] '
    'OR "site-directed mutagenesis"[All Fields] OR "functional analysis"[All Fields] '
    'OR "phenotypic characterization"[All Fields] OR "gene deletion"[All Fields] '
    'OR "insertional mutagenesis"[All Fields] OR "reverse genetics"[All Fields] '
    'OR "complementation assay"[All Fields])'
)

full_query = f'{AMR_TERMS} AND "pubmed pmc"[sb] {PUBTYPE_EXCLUSIONS}'

print(full_query)

**Discuss before running anything**: if you were building a corpus for your own project today, how many years back would you go — 5, 10, 25? Trade-offs:
- **Recall vs. noise** — wider ranges catch more papers but pull in more studies using outdated nomenclature/methods.
- **PMC/BioC availability** — `"pubmed pmc"[sb]` restricts to records with PMC full text at all; very old papers are disproportionately PubMed-only (abstract, no full text).
- **Publication-type drift** — case reports and reviews are a bigger share of older AMR literature.
- **Your research question** — tracking how resistance nomenclature evolved argues for a wide range *deliberately*; building an eval set for a current pipeline argues for a narrower, more homogeneous one.

In [ ]:
from Bio import Entrez

Entrez.email = ENTREZ_EMAIL
if NCBI_API_KEY:
    Entrez.api_key = NCBI_API_KEY

# NCBI rate limit: ~3 req/s without an API key, ~10 req/s with one (see Section 0's secrets cell).
NCBI_SLEEP = 0.11 if NCBI_API_KEY else 0.4


start_year, end_year = 2025, 2026
demo_retmax = 20

handle = Entrez.esearch(
    db="pubmed",
    term=full_query,
    datetype="PDAT",
    mindate=str(start_year),
    maxdate=str(end_year),
)
    #retmax=demo_retmax,

search_result = Entrez.read(handle)
handle.close()

demo_pmids = search_result["IdList"]
print(f"Query matched {search_result['Count']} PubMed records in {start_year}-{end_year} "
      f"(fetched first {len(demo_pmids)}):")
print(demo_pmids)

### c) Excluding retracted papers and papers that refer to a retraction

`esearch` alone has no clean way to exclude retractions and linked Expression-of-Concern notices in one
query-level term, so this does it as a **second pass** over full records fetched with `efetch`:

1. For every matched PMID, fetch `PublicationTypeList` and `CommentsCorrectionsList`.
2. Flag a PMID as "drop" if its own `PublicationType` includes `Retracted Publication` / `Retraction Notice` /
   `Expression of Concern` — the record is itself a retraction/EoC notice.
3. Also flag any PMID *referenced* by such a record via `CommentsCorrections/@RefType` of `RetractionOf`,
   `RetractionIn`, `ExpressionOfConcernFor`, or `ExpressionOfConcernIn` — this is the paper **being**
   retracted/flagged, which may not say so itself in `PublicationType`.
4. Drop the union of both sets from the result.

This is implemented below in Python (`efetch` + `Entrez.read`), batched to stay under NCBI's URL-length
limits for large PMID lists.

In [ ]:
# NOTE: the following cell runs ~len(all_pmids)/200 efetch calls at NCBI_SLEEP seconds each
# for the full 56k-record corpus that's still a long time even with an API key.
# set the flag below to run on all  (`True`) or randomly picked 200 PMIDs (`False`)
RUN_ON_FULL_SET = False  # set True to check every one of the `all_pmids` records

In [ ]:
import time
from datetime import date, timedelta
from urllib.error import HTTPError, URLError
from http.client import IncompleteRead

RETRACTION_TYPES = {"Retracted Publication", "Retraction Notice", "Expression of Concern"}
RETRACTION_REFTYPES = {"RetractionOf", "RetractionIn", "ExpressionOfConcernFor", "ExpressionOfConcernIn"}


def entrez_call(fn, *args, max_retries=5, **kwargs):
    """Runs an Entrez.esearch/efetch call + Entrez.read with retry/backoff.

    NCBI connections occasionally drop mid-response on long-running loops (chunked-transfer
    read errors, HTTP 429/5xx) -- these are transient, not query bugs, so retry instead of
    letting one flaky request kill a multi-hour fetch."""
    for attempt in range(max_retries):
        try:
            handle = fn(*args, **kwargs)
            result = Entrez.read(handle)
            handle.close()
            return result
        except (HTTPError, URLError, IncompleteRead, ValueError) as e:
            wait = 2.0 * (2 ** attempt)
            print(f"  Entrez call failed ({e!r}), retrying in {wait:.0f}s "
                  f"(attempt {attempt + 1}/{max_retries})...")
            time.sleep(wait)
    raise RuntimeError(f"Entrez call failed after {max_retries} retries: {fn} {args} {kwargs}")


def find_retracted_pmids(pmids, batch_size=200):
    """Flags any PMID that is itself a retraction/Expression-of-Concern notice, or that is
    referenced by one (i.e. the paper being retracted/flagged, which may not say so in its
    own PublicationType)."""
    drop = set()

    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]
        records = entrez_call(Entrez.efetch, db="pubmed", id=batch, rettype="xml", retmode="xml")

        for article in records["PubmedArticle"]:
            citation = article["MedlineCitation"]
            pmid = str(citation["PMID"])

            pub_types = {str(t) for t in citation["Article"].get("PublicationTypeList", [])}
            if pub_types & RETRACTION_TYPES:
                drop.add(pmid)

            for correction in article.get("PubmedData", {}).get("CommentsCorrectionsList", []):
                ref_type = correction.attributes.get("RefType")
                if ref_type in RETRACTION_REFTYPES:
                    drop.add(pmid)  # this record itself refers to a retraction/EoC
                    linked_pmid = correction.get("PMID")
                    if linked_pmid:
                        drop.add(str(linked_pmid))  # the paper being retracted/flagged

        time.sleep(NCBI_SLEEP)  # ~3 req/s anonymous, ~10 req/s with NCBI_API_KEY set
        if (i // batch_size) % 10 == 0:
            print(f"  ...checked {min(i + batch_size, len(pmids))}/{len(pmids)}")

    return drop


# PubMed's esearch hard-caps retstart at 9998 -- there is NO way to page past the first 9,999
# records of a single query via retstart/WebEnv (NCBI's own docs point this out and recommend
# EDirect, which works around it by date-bisecting). We do the same thing here: recursively
# split the date range into smaller windows until each window's match count is <=9999, then
# union the PMIDs across windows. No retstart paging is used at all.
ESEARCH_CAP = 9999


def fetch_pmids_in_range(term, start_date, end_date, depth=0):
    result = entrez_call(
        Entrez.esearch,
        db="pubmed",
        term=term,
        datetype="PDAT",
        mindate=start_date.strftime("%Y/%m/%d"),
        maxdate=end_date.strftime("%Y/%m/%d"),
        retmax=ESEARCH_CAP,
    )
    time.sleep(NCBI_SLEEP)

    count = int(result["Count"])
    if count <= ESEARCH_CAP:
        return result["IdList"]

    if start_date >= end_date:
        # Can't split a single day any further -- NCBI's cap means some records on this
        # exact day are unreachable via esearch. Rare for a title/abstract query; flag it.
        print(f"WARNING: {count} records on {start_date} alone exceed the {ESEARCH_CAP} cap; "
              f"only the first {ESEARCH_CAP} for that day are included.")
        return result["IdList"]

    mid_date = start_date + (end_date - start_date) // 2
    print(f"{'  ' * depth}{start_date}..{end_date}: {count} hits > cap, splitting at {mid_date}")
    left = fetch_pmids_in_range(term, start_date, mid_date, depth + 1)
    right = fetch_pmids_in_range(term, mid_date + timedelta(days=1), end_date, depth + 1)
    return left + right


print(f"Fetching all PMIDs for {full_query!r} in {start_year}-{end_year} via date-bisection...")
all_pmids = fetch_pmids_in_range(full_query, date(start_year, 1, 1), date(end_year, 12, 31))
all_pmids = list(dict.fromkeys(all_pmids))  # de-dup while preserving order (adjacent windows can't overlap, but be safe)
print(f"Collected {len(all_pmids)} PMIDs total.")


import random
pmids_to_check = all_pmids if RUN_ON_FULL_SET else random.sample(all_pmids, min(200, len(all_pmids)))

print(f"Checking {len(pmids_to_check)} PMIDs for retraction / Expression-of-Concern status...")
retracted_or_flagged = find_retracted_pmids(pmids_to_check)
clean_pmids = [p for p in pmids_to_check if p not in retracted_or_flagged]

print(f"Dropped {len(retracted_or_flagged)} retracted/Expression-of-Concern PMIDs "
      f"({len(pmids_to_check)} checked -> {len(clean_pmids)} clean).")
if retracted_or_flagged:
    print("Dropped PMIDs:", sorted(retracted_or_flagged))

### Optional check

How many of the PMIDs covered in the AMR portal are present in this list of PMIDs obtained after all checks.

In [ ]:
import pandas as pd
from Bio import Entrez
import os

# 1. Load the PMIDs from amr_pmids_unique.csv inside the workshop_data folder
csv_path = "workshop_data/amr_pmids_unique.csv"
if os.path.exists(csv_path):
    csv_df = pd.read_csv(csv_path)
    print(f"Loaded {len(csv_df)} records from {csv_path}.")

    # Get unique PMIDs from CSV as strings
    csv_pmids = [str(x) for x in csv_df.iloc[:, 0].dropna().unique()]

    # 2. Check which CSV PMIDs match our search query in PubMed
    # We do this by searching for the PMIDs AND the search query in batches to avoid URL length limits
    print("Checking which PMIDs from the CSV match the search query...")
    matching_pmids = []
    batch_size = 300  # Safe batch size for URL queries

    for i in range(0, len(csv_pmids), batch_size):
        pmid_batch = csv_pmids[i:i + batch_size]
        pmid_query = " OR ".join([f"{pmid}[pmid]" for pmid in pmid_batch])
        combined_query = f"({full_query}) AND ({pmid_query})"

        try:
            handle = Entrez.esearch(
                db="pubmed",
                term=combined_query,
                datetype="PDAT",
                mindate=str(start_year),
                maxdate=str(end_year),
                retmax=batch_size
            )
            search_results = Entrez.read(handle)
            handle.close()
            matching_pmids.extend(search_results["IdList"])
        except Exception as e:
            print(f"Error processing batch starting at index {i}: {e}")

    print(f"\nResults:\n{len(matching_pmids)} out of the {len(csv_df)} CSV PMIDs match the {start_year}-{end_year} PubMed query.")
    if len(matching_pmids) > 0:
        print("Sample of matching PMIDs:", matching_pmids[:10])
else:
    print(f"\nError: '{csv_path}' not found in the workshop_data directory.")